In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import json
from pathlib import Path
from tqdm.notebook import tqdm
from pprint import pprint
import pickle
import networkx as nx
import matplotlib.pyplot as plt
# import igraph as ig
import torch
from torch_geometric.data import Data

from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch.nn.functional as F

2025-03-03 07:47:38.431133: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-03 07:47:38.439496: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-03 07:47:38.459058: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740988058.492955  709817 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740988058.503120  709817 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regist

## Protein

In [ ]:
# Create a dataframe
df = pd.read_csv('data/kg.csv', low_memory=False)
print(df.shape)
df.head()

(8100498, 12)


,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
0,protein_protein,ppi,0,9796,gene/protein,PHYHIP,NCBI,8889,56992,gene/protein,KIF15,NCBI
1,protein_protein,ppi,1,7918,gene/protein,GPANK1,NCBI,2798,9240,gene/protein,PNMA1,NCBI
2,protein_protein,ppi,2,8233,gene/protein,ZRSR2,NCBI,5646,23548,gene/protein,TTC33,NCBI
3,protein_protein,ppi,3,4899,gene/protein,NRF1,NCBI,11592,11253,gene/protein,MAN1B1,NCBI
4,protein_protein,ppi,4,5297,gene/protein,PI4KA,NCBI,2122,8601,gene/protein,RGS20,NCBI


In [3]:
nodes = pd.concat([
    df.get(['x_id', 'x_type', 'x_name', 'x_source']).rename(columns={
        'x_id': 'node_id', 'x_type': 'node_type', 'x_name': 'node_name', 'x_source': 'node_source'
        }),
    df.get(['y_id', 'y_type', 'y_name', 'y_source']).rename(columns={
        'y_id': 'node_id', 'y_type': 'node_type', 'y_name': 'node_name', 'y_source': 'node_source'
        })
    ])

nodes = nodes.drop_duplicates().reset_index().drop('index', axis=1).reset_index().rename(
    columns={'index': 'node_idx'})

edges = pd.merge(df, nodes, 'left', left_on=['x_id','x_type', 'x_name','x_source'], 
                 right_on=['node_id','node_type','node_name','node_source'])
edges = edges.rename(columns={'node_idx': 'x_idx'})
edges = pd.merge(edges, nodes, 'left', left_on=['y_id','y_type', 'y_name','y_source'], 
                 right_on=['node_id','node_type','node_name','node_source'])
edges = edges.rename(columns={'node_idx': 'y_idx'})

edge_index = edges.get(['x_idx', 'y_idx']).values.T


In [5]:
graph = ig.Graph()
graph.add_vertices(list(range(nodes.shape[0])))
graph.add_edges([tuple(x) for x in edge_index.T])
G = nx.Graph()
G = graph.to_networkx()

In [7]:
print(G)

MultiGraph with 129375 nodes and 8100498 edges


In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.9)
labels = nx.get_edge_attributes(G, 'label')
plt.figure(figsize=(12, 10))
nx.draw(G, pos, with_labels=True, font_size=10, node_size=700, node_color='lightblue', edge_color='gray', alpha=0.6)
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=8, label_pos=0.3, verticalalignment='baseline')
plt.title('Knowledge Graph')
plt.show()

In [ ]:
degree_centrality = nx.degree_centrality(G)
for node, centrality in degree_centrality.items():
    print(f'{node}: Degree Centrality = {centrality:.2f}')

In [11]:
# Calculate centrality measures
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)

# Visualize centrality measures
plt.figure(figsize=(15, 10))

# Degree centrality
plt.subplot(131)
nx.draw(G, pos, with_labels=True, font_size=10, node_size=[v * 3000 for v in degree_centrality.values()], node_color=list(degree_centrality.values()), cmap=plt.cm.Blues, edge_color='gray', alpha=0.6)
plt.title('Degree Centrality')

# Betweenness centrality
plt.subplot(132)
nx.draw(G, pos, with_labels=True, font_size=10, node_size=[v * 3000 for v in betweenness_centrality.values()], node_color=list(betweenness_centrality.values()), cmap=plt.cm.Oranges, edge_color='gray', alpha=0.6)
plt.title('Betweenness Centrality')

# Closeness centrality
plt.subplot(133)
nx.draw(G, pos, with_labels=True, font_size=10, node_size=[v * 3000 for v in closeness_centrality.values()], node_color=list(closeness_centrality.values()), cmap=plt.cm.Greens, edge_color='gray', alpha=0.6)
plt.title('Closeness Centrality')

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [ ]:
edge_index = torch.tensor(edge_index, dtype=torch.long)
data = Data(edge_index=edge_index)

In [ ]:
from torch_geometric.nn import Node2Vec

model = Node2Vec(data.edge_index, embedding_dim=16, walk_length=10, context_size=5, walks_per_node=10,
                    num_negative_samples=1, p=1, q=1, sparse=True)

In [9]:
loader = model.loader(batch_size=128, shuffle=True, num_workers=4)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
def train():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

for epoch in range(1, 101):
    loss = train()
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}')


KeyboardInterrupt: 

In [10]:
torch.cuda.is_available()

False

In [ ]:
from node2vec import Node2Vec
from sklearn.manifold import TSNE

node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=200, workers=4)
model = node2vec.fit(window=10, min_count=1, batch_words=4)

embeddings = np.array([model.wv[node] for node in G.nodes()])
tsne = TSNE(n_components=2, perplexity=10, max_iter=400)
embeddings_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(12, 10))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c='blue', alpha=0.7)

# Add node labels
for i, node in enumerate(G.nodes()):
    plt.text(embeddings_2d[i, 0], embeddings_2d[i, 1], node, fontsize=8)
plt.title('Node Embeddings Visualization')
plt.show()

Computing transition probabilities:   0%|          | 0/129375 [00:00<?, ?it/s]

In [ ]:
from sklearn.cluster import KMeans

num_clusters = 3
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)

plt.figure(figsize=(6, 5))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=cluster_labels, cmap=plt.cm.Set1, alpha=0.7)
plt.legend(handles=plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                               c=cluster_labels, cmap=plt.cm.Set1, alpha=0.7).legend_elements()[0], 
                               labels=[f'Cluster {i}' for i in range(num_clusters)], title='Cluster Label')

for i, node in enumerate(G.nodes()):
    plt.text(embeddings_2d[i, 0], embeddings_2d[i, 1], node, fontsize=8)

plt.title('K-Means Clustering in Embedding Space with Node Labels')
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
nx.draw(G, pos, with_labels=True, font_size=10, node_size=700, node_color=cluster_labels, 
        cmap=plt.cm.Set1, edge_color='gray', alpha=0.6)
plt.title('Graph Clustering using K-Means')

plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

# Perform DBSCAN clustering on node embeddings
dbscan = DBSCAN(eps=1.0, min_samples=2) # Adjust eps and min_samples
cluster_labels = dbscan.fit_predict(embeddings)

# Visualize clusters
plt.figure(figsize=(6, 5))
nx.draw(G, pos, with_labels=True, font_size=10, node_size=700, node_color=cluster_labels, cmap=plt.cm.Set1, edge_color='gray', alpha=0.6)
plt.title('Graph Clustering using DBSCAN')
plt.show()

## Load Graph

In [ ]:
tokenizer = T5Tokenizer.from_pretrained('t5-base')
model = T5ForConditionalGeneration.from_pretrained('t5-base')

2025-03-02 09:52:25.806166: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-02 09:52:25.814431: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-02 09:52:25.834334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740909145.868662  655428 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740909145.879066  655428 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regist

In [111]:
data_path = Path('data') / 'squad_graph'
data = json.loads((data_path / 'dev-v2.0_graph.json').read_text())

In [129]:
def get_items(data, G, tokenizer=tokenizer, model=model):
    first_token = True
    first_token_item = None
    prev_item = None
    edge = None
    is_node = True
    # print(data)
    for token in data:
        token = token.strip()
        if token == '':
            continue
        token = token.replace('__no_node__', '')
        token = token.replace('Question: ', '')
        token = tokenizer.encode(token, return_tensors='pt')
        if is_node:
            # remove the string __no_node__ from the token
            if token not in G:
                G.add_node(token)
            # print(token)
            if first_token:
                first_token_item = token
                first_token = False
            if prev_item is None:
                prev_item = token
            else:
                G.add_edge(prev_item, token, label=edge)
                prev_item = token
        else:
            edge = token
        is_node = not is_node
    return first_token_item

In [ ]:
G = nx.Graph()
answer_tensor = tokenizer.encode('answer', return_tensors='pt')
plausible_answer_tensor = tokenizer.encode('plausible_answer', return_tensors='pt')
context_tensor = tokenizer.encode('context', return_tensors='pt')

prog_bar = tqdm(enumerate(data['data']), total=len(data['data']))
for i, item in prog_bar:
    for j, paragraph in enumerate(item['paragraphs']):
        start_tokens = []
        for k, qa in enumerate(paragraph['qas']):
            is_impossible = qa['is_impossible']
            question = qa['question']
            question_start_tokens = []
            # if len(question) > 1:
            #     print('q', question)
            for q_set in question:
                first_token_item = get_items(q_set, G)
                start_tokens.append(first_token_item)
                question_start_tokens.append(first_token_item)
            
            if 'answers' not in qa:

                for ans in qa['plausible_answers']:
                    answer = ans['text']
                    if isinstance(answer, str):
                        answer = tokenizer.encode(answer, return_tensors='pt')
                        G.add_node(answer)
                        for qst in question_start_tokens:
                            G.add_edge(qst, answer, label=plausible_answer_tensor)
                        start_tokens.append(answer)
                    else:
                        for a_set in answer:
                            first_token_item = get_items(a_set, G)
                            for qst in question_start_tokens:
                                G.add_edge(qst, first_token_item, label=plausible_answer_tensor)
                            start_tokens.append(first_token_item)
            else:
                for ans in qa['answers']:
                    answer = ans['text']
                    if isinstance(answer, str):
                        answer = tokenizer.encode(answer, return_tensors='pt')
                        G.add_node(answer)
                        for qst in question_start_tokens:
                            G.add_edge(qst, answer, label=answer_tensor)
                        start_tokens.append(answer)
                    else:
                        for a_set in answer:
                            first_token_item = get_items(a_set, G)
                            start_tokens.append(first_token_item)
                            for qst in question_start_tokens:
                                G.add_edge(qst, first_token_item, label=answer_tensor)
        p = paragraph['context'][0]
        # if len(p) > 1:
        #     print('p', p)
        for p_set in p:
            first_token_item = get_items(p_set, G)
            start_tokens.append(first_token_item)
            for st in set(start_tokens):
                # print(st)
                G.add_edge(st, first_token_item, label=context_tensor)         
        # break

  0%|          | 0/35 [00:00<?, ?it/s]

In [134]:
print(G)

Graph with 52696 nodes and 83245 edges


In [135]:
# print a few nodes and edges
print(list(G.nodes())[:5])
print(list(G.edges(data=True))[:5])

[tensor([[13615,    26,    63,     1]]), tensor([[4073,  684,    1]]), tensor([[1410,    1]]), tensor([[1410,    1]]), tensor([[1410,    1]])]
[(tensor([[13615,    26,    63,     1]]), tensor([[4073,  684,    1]]), {'label': tensor([[4567, 3023,    1]])}), (tensor([[13615,    26,    63,     1]]), tensor([[1410,    1]]), {'label': tensor([[1525,    1]])}), (tensor([[13615,    26,    63,     1]]), tensor([[1410,    1]]), {'label': tensor([[1525,    1]])}), (tensor([[13615,    26,    63,     1]]), tensor([[1410,    1]]), {'label': tensor([[1525,    1]])}), (tensor([[13615,    26,    63,     1]]), tensor([[1410,    1]]), {'label': tensor([[1525,    1]])})]


In [ ]:
# Save the graph

# pickle.dump(G, open('data/squad_graph/dev-v2.0_graph.pickle', 'wb'))

In [3]:
# load the graph
G = pickle.load(open('data/squad_graph/dev-v2.0_graph.pickle', 'rb'))

In [ ]:
def get_string(dct):
    words = []
    for k, v in dct.items():
        words.append(k + ': ')
        if isinstance(v, dict):
            words += get_string(v)
        else:
            words += v.split()
    return words

def convert_to_graph(dct):
    words = get_string(dct)
    edge_index = [[i, i + 1] for i in range(len(words) - 1)]
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    x = []
    for word in words:
        x.append(tokenizer.encode(word, return_tensors='pt')[0].to(dtype=torch.long))
    # pad the sequences
    max_len = max([len(xi) for xi in x])
    x = [F.pad(xi, (0, max_len - len(xi))) for xi in x]
    x = torch.stack(x)
    return Data(x=x, edge_index=edge_index.t().contiguous())

In [ ]:
data_path = Path.cwd() / 'data/squad'
files = list(data_path.glob('*'))
res_data = []
with open(files[1], 'r') as f:
    data = json.load(f)
    question_count = 0
    prog_bar = tqdm(enumerate(data['data']), total=len(data['data']))
    for i, item in prog_bar:
        new_dc = {'title': item['title'], 'paragraphs': []}
        for j, paragraph in enumerate(item['paragraphs']):
            questions = []
            for k, qa in enumerate(paragraph['qas']):
                is_impossible = qa['is_impossible']
                question = qa['question']
                question_count += 1
                prog_bar.set_description(' Question {} Question Count {}'.format( question, question_count), refresh=True)
                answers = []
                plausible_answers = []
                if len(qa['answers']) == 0:
                    for ans in qa['plausible_answers']:
                        
                        answer = ans['text']
                        answer_start = ans['answer_start']
                        plausible_answers.append({'text': answer, 'answer_start': answer_start})
                else:
                    for ans in qa['answers']:
                        answer = ans['text']
                        answer_start = ans['answer_start']
                        answers.append({'text': answer, 'answer_start': answer_start})
                if len(answers) > 0:
                    questions.append({'question': question, 'answers': answers, 'is_impossible': is_impossible})
                else:
                    questions.append({'question': question, 'plausible_answers': plausible_answers, 'is_impossible': is_impossible})
            p = paragraph['context']
            
            new_dc['paragraphs'].append({'context': p, 'qas': questions})
        res_data.append(new_dc)

Processing dev-v2.0:   0%|          | 0/442 [00:00<?, ?it/s]

In [31]:
dset = []
for item in res_data:
    dset_dct = {}
    outputs = {}
    for paragraph in item['paragraphs']:
        dset_dct['context'] = paragraph['context']
        
        for qa in paragraph['qas']:
            dset_dct['question'] = qa['question']
            if qa['is_impossible']:
                outputs['answers'] = qa['plausible_answers']
            else:
                outputs['answers'] = qa['answers']
            outputs['is_impossible'] = qa['is_impossible']
            dset.append((convert_to_graph(dset_dct), outputs))

KeyboardInterrupt: 

## Load Data

In [ ]:
train_data = pickle.load(open(Path('data') / 'squad' / 'train_dset.pkl', 'rb'))

In [18]:
print(len(train_data), len(train_data[0]), len(train_data[0][0]))
train_data[0]

130319 2 2


(Data(x=[117, 13], edge_index=[2, 116]),
 {'answers': [{'text': 'Salma Hayek and Frida Giannini', 'answer_start': 533}],
  'is_impossible': False})

In [27]:
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple, Dict, Any
import pytorch_lightning as pl

class EdgePredictionDataset(Dataset):
    def __init__(self, data: List[Tuple[Data, Dict[str, Any]]]):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]
    

def collate_fn(batch):
    data_list, outputs_list = zip(*batch)
    # batch_data = Data.from_data_list(data_list)
    return data_list, outputs_list

class EdgePredictionDataModule(pl.LightningDataModule):
    def __init__(self, train_data: List[Tuple[Data, Dict[str, Any]]], 
                 val_data: List[Tuple[Data, Dict[str, Any]]],
                 batch_size: int = 32):
        super().__init__()
        self.data = train_data
        self.val_data = val_data
        self.batch_size = batch_size
    
    def setup(self, stage=None):
        self.train_data = EdgePredictionDataset(self.data)

        if self.val_data is not None:
            self.val_data = EdgePredictionDataset(self.val_data)
    
    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True, collate_fn=collate_fn)

    def val_dataloader(self):
        return DataLoader(self.val_data, batch_size=self.batch_size, collate_fn=collate_fn)

data_module = EdgePredictionDataModule(train_data, None, batch_size=32)
data_module.setup()

In [31]:
for batch in data_module.train_dataloader():
    print(len(batch[0]))
    print(len(batch[1]))
    print(len(batch))
    break

32
32
2


In [ ]:
from models.gnns import GCN
import torch
from torch_geometric.data import Data, DataLoader

model = GCN(in_channels=tokenizer.vocab_size, hidden_channels=64, out_channels=tokenizer.vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1, 101):
    model.train()
    for data, outputs in dset:
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        print(out)
        break
        loss = F.cross_entropy(out, outputs['answers'])
        loss.backward()
        optimizer.step()
    break
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}')
    